In [12]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from datetime import date, timedelta
import time


In [13]:
START_DATE = date(2025, 1, 1)
END_DATE   = date(2025, 1, 31)

BASE_URL = "https://kworb.net/spotify/world_daily.html?date={}"

rows = []

In [16]:
current = START_DATE
while current <= END_DATE:
    date_str = current.strftime("%Y-%m-%d")
    url = BASE_URL.format(date_str)

    try:
        r = requests.get(
            url,
            headers={"User-Agent": "Mozilla/5.0"},
            timeout=15
        )

        soup = BeautifulSoup(r.text, "html.parser")
        tables = soup.find_all("table")

        if not tables:
            print(f"⚠️ No tables {date_str}")
            current += timedelta(days=1)
            continue

        # Pick the biggest table = chart table
        table = max(tables, key=lambda t: len(t.find_all("tr")))

        for tr in table.find_all("tr")[1:]:
            tds = tr.find_all("td")
            if len(tds) < 6:
                continue

            artist_title = tds[2].text.strip()
            daily_streams = int(tds[5].text.replace(",", ""))

            if " - " in artist_title:
                artist, track = artist_title.split(" - ", 1)
            else:
                artist = "Unknown"
                track = artist_title

            rows.append([
                date_str,
                track.strip(),
                artist.strip(),
                daily_streams
            ])

        print(f"✅ {date_str}")
        time.sleep(1)

    except Exception as e:
        print(f"❌ Error {date_str}: {e}")

    current += timedelta(days=1)

# -----------------------------
# MONTHLY AGGREGATION
# -----------------------------

df = pd.DataFrame(rows, columns=["date", "track", "artist", "streams"])
df["date"] = pd.to_datetime(df["date"])
df["month"] = df["date"].dt.to_period("M")

monthly = (
    df.groupby(["month", "track", "artist"])["streams"]
    .sum()
    .reset_index()
)

top_monthly = (
    monthly
    .sort_values(["month", "streams"], ascending=[True, False])
    .groupby("month")
    .head(1)
)

top_monthly.to_csv(
    "spotify_global_monthly_top_2025.csv",
    index=False
)

print("\n🎉 DONE")
print("📁 spotify_global_monthly_top_2025.csv created")

⚠️ No tables 2025-01-01
⚠️ No tables 2025-01-02
⚠️ No tables 2025-01-03


KeyboardInterrupt: 